# World Cup Transit Service near SoFi Stadium

In [1]:
import warnings
warnings.filterwarnings("ignore")
                        
import altair as alt
import folium
import geopandas as gpd
import google.auth
import pandas as pd

import world_cup_vars as wc_vars
import D1_prep_trips as D1
import D2_prep_stop_arrivals as D2
import chart_utils

credentials, _ = google.auth.default()

## Regional Trips

In [2]:
sofi_trips = D1.filter_fct_daily_schedule_rt_route_direction_summary_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    route_name_dict = wc_vars.special_socal_routes_dict,
    event_time_of_day_dict = wc_vars.sofi_match_times  
)

In [3]:
daily_trips_by_operator = D1.aggregate_daily_trips(
    sofi_trips, ["service_date", "schedule_name"])

In [4]:
# just add the before and after, set to zero, otherwise LA Metro Events 
# single point drops away from chart visibility
append_la_metro_events = pd.DataFrame({
    'service_date': ["2026-07-09", "2026-07-11"],
})
append_la_metro_events = append_la_metro_events.assign(
    service_date = pd.to_datetime(append_la_metro_events.service_date).dt.normalize(),
    schedule_name = "LA Metro Events Schedule",
    n_trips = 0
)

daily_trips_by_operator2 = pd.concat(
    [daily_trips_by_operator, append_la_metro_events], 
    axis=0, ignore_index=True
)

In [5]:
chart_utils.trip_chart_with_event_dates(
    daily_trips_by_operator2, wc_vars.sofi_dates, color_col="schedule_name"
).properties(
    title= "Daily Trips by Operator during World Cup",
    width=500, height=300
)

alt.LayerChart(...)

## Trips by Route

In [6]:
daily_trips_by_route = D1.aggregate_daily_trips(
    sofi_trips, ["service_date", "schedule_name", "route_name"]
)

In [7]:
chart_utils.trip_chart_with_event_dates(
    daily_trips_by_route, wc_vars.sofi_dates, color_col="route_name"
).properties(
    title= "Daily Trips by Route during World Cup",
    width=500, height=300
)

alt.LayerChart(...)

## LA Metro World Cup Feed
Thicker routes have more trips!

In [8]:
la_metro_events = sofi_trips[sofi_trips.schedule_name == "LA Metro Events Schedule"].reset_index(drop=True)

In [9]:
#https://matplotlib.org/stable/users/explain/colors/colormaps.html
la_metro_events[["route_name", "direction_id", "geometry", "n_trips"]].explore(
    "route_name",
    tiles = "CartoDB Positron",
    style_kwds={"style_function": lambda x: {"weight":x["properties"]["n_trips"]*0.05}},
    cmap="Set1"
)

## Map of Routes for all other feeds
* map of routes (how to compare event vs non-event)?
   * aggregate daily trips on event days (weekday + weekend), aggregate daily trips on non-event days (weekday + weekend)
   * show the difference? 
* can we get stops that show up on routes as a layer too?
* see a chart that filters for that route, show weekday + weekend event vs non-event trips?
* can be combined with World Cup feed

In [10]:
trips_by_event = sofi_trips.groupby(
    ["schedule_name", "event_day", "day_type", "route_name", "route_type"]
).agg({
    "n_trips": "sum",
    "service_date": "nunique"
}).reset_index().rename(columns = {
    "service_date": "n_days"
})

trips_by_event = trips_by_event.assign(
    daily_trips = trips_by_event.n_trips.divide(trips_by_event.n_days).round(1)
)

In [11]:
trips_by_event_wide = D2.make_wide(
    trips_by_event,
    group_cols = ["schedule_name", "day_type", "route_name"],
    metric_cols = ["daily_trips"]
)

In [12]:
selection = alt.selection_point(fields=["schedule_name"], bind="legend")
alt.Chart(trips_by_event).mark_bar().encode(
    x="daily_trips",
    y=alt.Y("event_day", title = ""),
    row=alt.Row("day_type:N", title = ""),
    color=alt.Color("schedule_name:N"), 
    opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.1)),
).add_params(selection).transform_filter(selection)

alt.Chart(...)

In [13]:
alt.Chart(trips_by_event).mark_bar().encode(
    x="daily_trips",
    y=alt.Y("event_day", title = ""),
    column=alt.Column("day_type:N", title = ""),
    row=alt.Row("schedule_name", title=""),
    color=alt.Color("schedule_name:N"), 
).properties(width=150, height=50)

alt.Chart(...)

In [14]:
selection = alt.selection_point(fields=["schedule_name"], bind="legend")
alt.Chart(trips_by_event_wide).mark_bar().encode(
    x="change_daily_trips",
    y="route_name:N",
    column=alt.Column("day_type:N", title=""),
    color=alt.Color("schedule_name:N"), 
    opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.1)),
).add_params(selection)

alt.Chart(...)

## Stop Arrivals

In [ ]:
sofi_stop_arrivals = D2.filter_fct_daily_scheduled_stops_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    route_name_dict = wc_vars.special_socal_routes_dict,
    event_time_of_day_dict = wc_vars.sofi_match_times
)

# Do this separately, because we need stop's pt geom
arrivals_by_event_type = D2.aggregate_by_event_type(sofi_stop_arrivals)

In [ ]:
weekday_wide = D2.make_wide(
    arrivals_by_event_type[arrivals_by_event_type.day_type == "weekday"],
    group_cols = ["schedule_name", "stop_id", "stop_name"],
    metric_cols = ["daily_arrivals"]
).rename(columns = {
    **{c: f"weekday_{c}" for c in ["daily_arrivals_event", "daily_arrivals_non_event", "change_daily_arrivals"]}
})

weekend_wide = D2.make_wide(
    arrivals_by_event_type[arrivals_by_event_type.day_type == "weekend"],
    group_cols = ["schedule_name", "stop_id", "stop_name"],
    metric_cols = ["daily_arrivals"]
).rename(columns = {
    **{c: f"weekend_{c}" for c in ["daily_arrivals_event", "daily_arrivals_non_event", "change_daily_arrivals"]}
})

arrivals_wide = pd.merge(
    weekday_wide,
    weekend_wide,
    on = ["schedule_name", "stop_id", "stop_name"],
    how = "inner"
).pipe(D2.merge_in_stop_geom, sofi_stop_arrivals)

In [ ]:
operator_df = (
    arrivals_wide
    .groupby(["schedule_name", "route_id_array", "stop_name"])
    .agg({
        "weekday_change_daily_arrivals": "sum",
        "weekend_change_daily_arrivals": "sum",
        "stop_id": "nunique"
    })
    .reset_index()
    .rename(columns = {"stop_id": "n_stop_ids"})
)

In [ ]:
for i in sorted(operator_df.schedule_name.unique()):
    chart = chart_utils.weekday_weekend_chart_by_operator(operator_df, i)
    display(chart)